<a href="https://colab.research.google.com/github/BridgingAISocietySummerSchools/Python-for-Data-Science-Workshop/blob/main/02_advanced_self_learning/05_functions_and_modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📓 Notebook 5 — Functions and Modules

> **Advanced & Self-Learning Track — Notebook 5 of 14** · **Estimated time:** 35–40 min · **Difficulty:** Beginner
>
> This track is the self-paced deep-dive of the course. If you are looking for the 90-minute live **Introduction Session**, it lives in [`01_introduction/`](../01_introduction/).

### 🔗 Building on Track 1

The 90-minute live Introduction Session covered functions in [`../01_introduction/04_functions.ipynb`](../01_introduction/04_functions.ipynb): why functions exist, `def`, parameters, `return`, `return` vs `print`, and a single default argument. This notebook revisits that ground briefly and then goes much deeper — keyword arguments, `*args`/`**kwargs`, scope and mutable arguments, docstrings, type hints, lambdas, and `import`. A cold start is fine: Sections 1–3 build the basics up from zero, so treat them either as revision or as your first encounter with functions.

A function is a *reusable block of code with a name*. Whenever you find yourself copy-pasting the same handful of lines, you should wrap them in a function. In data science, functions are how you build clean, testable, *modular* pipelines — and `import` is how you tap into the ocean of pre-built tools (numpy, pandas, scikit-learn, …).

## 🎯 Learning objectives

By the end of this notebook you will:

1. Define and call functions with positional and keyword arguments.
2. Use **default arguments**, `*args`, and `**kwargs`.
3. Return single and multiple values — and explain why `return` beats `print`.
4. Understand **scope** — what is local, what is global, and why mutating arguments is risky.
5. Use **lambda** expressions for short throwaway functions.
6. Read and write **docstrings** and apply basic **type hints**.
7. Import functions from the standard library and third-party packages.

## ✅ Prerequisites

Notebooks 1–4 of this track ([`01_python_basics.ipynb`](01_python_basics.ipynb) through [`04_dictionaries_advanced.ipynb`](04_dictionaries_advanced.ipynb)): variables, control flow, lists and comprehensions, dictionaries. **No pandas or NumPy required** — those arrive next, in Notebooks 6 and 7.


## 1. Why functions?

Here is how the pain usually starts in a real analysis. You clean and average temperature readings from sensor A, then copy-paste the block for sensors B and C:

```python
# Sensor A
valid_a = [x for x in sensor_a if x is not None and -50 <= x <= 60]
mean_a  = sum(valid_a) / len(valid_a)

# Sensor B — copy, paste, rename…
valid_b = [x for x in sensor_b if x is not None and -50 <= x <= 60]
mean_b  = sum(valid_b) / len(valid_b)

# Sensor C — copy, paste, rename…
valid_c = [x for x in sensor_c if x is not None and -50 <= x <= 60]
mean_c  = sum(valid_c) / len(valid_c)
```

Two weeks later you learn the upper limit should be **45, not 60**. You fix sensors A and B… and forget C. The notebook still runs happily — it just reports a **silently wrong number** for C. The copies have *diverged*, and nothing warns you.

A **function** puts the logic in exactly one place:

```python
def clean_mean(readings, low=-50, high=45):
    valid = [x for x in readings if x is not None and low <= x <= high]
    return sum(valid) / len(valid)

mean_a = clean_mean(sensor_a)
mean_b = clean_mean(sensor_b)
mean_c = clean_mean(sensor_c)   # C can no longer drift out of sync
```

The function version has a meaningful **name**, can be **tested** in isolation, can be **reused** without copy-paste, and — crucially — has **one** place to fix when requirements change.

## 2. Defining and calling a function

In [1]:
# The smallest useful function: no arguments, returns nothing
def greet():
    print("Hello, data scientist! 👋")

greet()
greet()                 # call as often as you like

Hello, data scientist! 👋
Hello, data scientist! 👋


In [2]:
# Functions become powerful once they take inputs and return outputs.
def square(x):
    return x * x        # `return` sends a value back to the caller

print(square(5))        # 25
result = square(7)
print(f"7 squared is {result}")

25
7 squared is 49


### `return` vs `print` — the #1 beginner bug

`print` *shows* a value on screen and then throws it away. `return` *hands the value back* to your code so you can keep computing with it. A single call looks identical either way — which is exactly why this bug is so sneaky.

> **Rule of thumb:** functions should `return` their result; `print` at the call site if you want to see it. If a variable mysteriously holds `None`, check whether the function prints instead of returns — or has no `return` at all (that also yields `None`).

In [3]:
def double_print(x):
    print(x * 2)            # shows the result on screen... but returns None

def double_return(x):
    return x * 2            # hands the result back to the caller

a = double_print(21)        # prints 42, but `a` receives nothing useful
b = double_return(21)       # prints nothing, but `b` now holds 42

print(f"a = {a}")           # a = None  ← the value was shown, then lost forever
print(f"b = {b}")           # b = 42    ← the value came back to us

print(b + 1)                # we can keep computing with b
# print(a + 1)              # ← uncomment, run, read the error: you cannot add 1 to None

42
a = None
b = 42
43


### Anatomy of a function definition

```
def   function_name(arg1, arg2=default):
│            │       │       │
│            │       │       └─ default value (optional)
│            │       └───────── parameter list
│            └───────────────── name (snake_case)
└────────────────────────────── the `def` keyword

    "docstring describing what the function does"
    body
    ...
    return something    # optional; default is None
```

## 3. Parameters with defaults

A parameter with a default value becomes **optional** when calling the function. This is the cleanest way to support "the common case is X, but you can override it".

In [4]:
def greet_user(name, greeting="Hello"):
    """Greet a user with an optional custom greeting."""
    return f"{greeting}, {name}!"

print(greet_user("Alice"))                       # uses the default
print(greet_user("Bob", greeting="Welcome"))     # overrides it
print(greet_user(name="Charlie"))                # positional → keyword

# Keyword arguments make calls self-documenting in long signatures
def normalize(values, lower=0.0, upper=1.0):
    lo, hi = min(values), max(values)
    return [(v - lo) / (hi - lo) * (upper - lower) + lower for v in values]

print(normalize([10, 20, 30, 40, 50]))
print(normalize([10, 20, 30, 40, 50], lower=-1, upper=1))

Hello, Alice!
Welcome, Bob!
Hello, Charlie!
[0.0, 0.25, 0.5, 0.75, 1.0]
[-1.0, -0.5, 0.0, 0.5, 1.0]


> ⚠️ **Mutable default trap.** Never write `def f(x=[]):`. The list is shared across calls. Use `def f(x=None): if x is None: x = []`.

## 4. Returning multiple values

A function can return a *tuple* of values; the caller usually unpacks it.

In [5]:
def summary_stats(values):
    """Return min, mean, and max as a tuple."""
    return min(values), sum(values)/len(values), max(values)

scores = [85, 92, 78, 96, 88, 73, 91]
lo, avg, hi = summary_stats(scores)
print(f"min={lo}, mean={avg:.2f}, max={hi}")

# Returning a dictionary is clearer when several values have distinct meanings
def summary_dict(values):
    return {
        "min":   min(values),
        "mean":  sum(values)/len(values),
        "max":   max(values),
        "count": len(values),
    }

print(summary_dict(scores))

min=73, mean=86.14, max=96
{'min': 73, 'mean': 86.14285714285714, 'max': 96, 'count': 7}


## 5. Variadic functions — `*args` and `**kwargs`

Open the documentation of almost any data-science function and you will meet these two symbols:

```python
plt.plot(*args, ..., **kwargs)                  # matplotlib
pd.read_csv(filepath_or_buffer, ..., **kwds)    # pandas
```

- `*args` collects any number of extra **positional** arguments into a tuple.
- `**kwargs` collects any number of extra **keyword** arguments into a dict.

They exist because a library author cannot know in advance how many arguments *you* will pass — and sometimes neither can you.

In [6]:
def total(*args):
    """Sum any number of numeric arguments."""
    return sum(args)

print(total(1, 2, 3))
print(total(10, 20, 30, 40, 50))

def describe(**kwargs):
    """Pretty-print every keyword argument."""
    for k, v in kwargs.items():
        print(f"  {k:<10}: {v}")

describe(name="Alice", age=20, role="student", gpa=3.8)

6
150
  name      : Alice
  age       : 20
  role      : student
  gpa       : 3.8


### You have been using `*args` all along — and the reverse trick

`print()` itself is variadic: it accepts any number of positional values. And the same symbols work **in the other direction** too — in a *call*, `*` unpacks a sequence and `**` unpacks a dict into individual arguments. That unpacking trick is everywhere in plotting code: bundle your style options in one dict, reuse them in every plot.

In [7]:
# print() collects any number of positional arguments with *args:
print("alpha", "beta", "gamma", sep=" | ")   # three args, one *args tuple

# The reverse: ** UNPACKS a dict into keyword arguments at the call site.
style = {"sep": " → ", "end": " 🏁\n"}
print("load", "clean", "model", **style)     # same as sep=" → ", end=" 🏁\n"

# ...and * unpacks a sequence into positional arguments:
def distance(x, y):
    return (x**2 + y**2) ** 0.5

point = (3, 4)
print(distance(*point))                      # unpacks into x=3, y=4 → 5.0

alpha | beta | gamma
load → clean → model 🏁
5.0


> 🔭 **Why this matters:** when `help(pd.read_csv)` or `help(plt.plot)` shows `**kwargs`, you can now read it as *"any extra keyword options are accepted and forwarded along"*. And in **Notebook 9 — Visualisation with Matplotlib** you will use the unpacking direction yourself: `plt.plot(x, y, **style)`.

## 6. Scope — what `x` means depends on *where*

Start with a failure you will hit sooner or later. This looks reasonable — but it crashes:

```python
def compute_stats(values):
    result = sum(values) / len(values)   # `result` is born here...

compute_stats([1, 2, 3])
print(result)      # NameError: name 'result' is not defined
```

`result` was created **inside** the function, so it *dies the moment the function returns*. Variables defined inside a function are **local** — they exist only during the call. Variables at the top level of a notebook (or module) are **global**. The only official door for a value to leave a function is `return` (which is why `compute_stats` should end with `return result`).

The reverse situation — the *same name* on both sides of the wall — behaves like this:

```
                    ┌────────────────── module / notebook scope ────────────────────┐
                    │  x = 10        ←  global x                                    │
                    │                                                                │
                    │  def f():                                                      │
                    │      x = 99    ←  a NEW, local x — does NOT touch the global  │
                    │      print(x)                                                  │
                    │                                                                │
                    │  f()           ← prints 99                                     │
                    │  print(x)      ← prints 10                                     │
                    └───────────────────────────────────────────────────────────────┘
```

In [8]:
x = 10                        # global

def show_x():
    x = 99                    # local — shadows the global one
    print(f"inside the function: x = {x}")

show_x()
print(f"outside the function: x = {x}")

inside the function: x = 99
outside the function: x = 10


> 🎯 **Best practice.** Pass everything you need *in* through parameters and return what you compute. Avoid the `global` keyword — it makes code hard to reason about.

There is one subtle catch with **mutable** objects:

In [9]:
def add_item(lst, item):
    lst.append(item)          # mutates the list IN PLACE
    return lst

numbers = [1, 2, 3]
add_item(numbers, 4)
print(numbers)                # [1, 2, 3, 4] — surprised?

# Lists are passed *by reference*. If you don't want the caller's list to
# change, make a copy inside the function:
def add_item_safe(lst, item):
    new = lst.copy()
    new.append(item)
    return new

numbers = [1, 2, 3]
result  = add_item_safe(numbers, 4)
print(f"original={numbers}, returned={result}")

[1, 2, 3, 4]
original=[1, 2, 3], returned=[1, 2, 3, 4]


## 7. Docstrings — describe what your function does

A **docstring** is a string literal placed as the first statement in a function body. Python, Jupyter, and most editors use it for inline help.

```python
def f(x):
    """Short one-line summary.

    Longer description, parameters, return values, examples.
    """
    ...
```

In [10]:
def safe_divide(a, b, default=0.0):
    """Divide a by b, returning `default` when b == 0.

    Parameters
    ----------
    a, b : numbers
    default : returned when b == 0 (default 0.0).

    Returns
    -------
    float
    """
    if b == 0:
        return default
    return a / b

# Editors and Jupyter expose this:
help(safe_divide)
print(safe_divide(10, 2))     # 5.0
print(safe_divide(10, 0))     # 0.0

Help on function safe_divide in module __main__:

safe_divide(a, b, default=0.0)
    Divide a by b, returning `default` when b == 0.
    
    Parameters
    ----------
    a, b : numbers
    default : returned when b == 0 (default 0.0).
    
    Returns
    -------
    float

5.0
0.0


## 8. Type hints — optional, but very helpful

Python is dynamically typed, but you can **annotate** your signature to document the types you expect:

```python
def mean(values: list[float]) -> float:
    return sum(values) / len(values)
```

Type hints do not affect runtime behaviour, but they:

- improve editor autocomplete and inline help,
- let tools like `mypy` catch type bugs before you run,
- serve as living documentation.

In [11]:
from typing import Iterable

def normalize(values: Iterable[float], lower: float = 0.0, upper: float = 1.0) -> list[float]:
    """Linearly rescale `values` into [lower, upper]."""
    values = list(values)
    lo, hi = min(values), max(values)
    if hi == lo:
        return [lower] * len(values)
    return [(v - lo) / (hi - lo) * (upper - lower) + lower for v in values]

print(normalize([10, 20, 30, 40, 50]))

[0.0, 0.25, 0.5, 0.75, 1.0]


## 9. Lambdas — tiny one-line functions

A **lambda** is an anonymous function. Use it where you would otherwise have to define a one-line `def` just to pass it somewhere (sorting, mapping, filtering).

In [12]:
# Sort employees by salary descending
employees = [
    {"name": "Alice",   "salary": 75000},
    {"name": "Bob",     "salary": 65000},
    {"name": "Charlie", "salary": 80000},
    {"name": "Diana",   "salary": 90000},
]

ranked = sorted(employees, key=lambda e: e["salary"], reverse=True)
for e in ranked:
    print(f"  {e['name']:<8} €{e['salary']:,}")

# Do not overuse lambdas — for anything longer than one line, give it a name.

  Diana    €90,000
  Charlie  €80,000
  Alice    €75,000
  Bob      €65,000


## 10. Importing — using other people's code

You will write this line thousands of times:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
```

Four flavours of import to recognise:

| Style                              | Use when                                            |
|------------------------------------|-----------------------------------------------------|
| `import math`                      | you want everything — call as `math.sqrt(...)`      |
| `import numpy as np`               | you want a shorter alias                            |
| `from math import sqrt, pi`        | you want a couple of specific names                 |
| `from math import *`               | ❌ almost never — it pollutes the namespace          |

In [13]:
# Standard library — ships with Python
import math
import random
import statistics

print(f"sqrt(2)  = {math.sqrt(2):.4f}")
print(f"pi       = {math.pi:.6f}")

random.seed(0)
print(f"random   = {random.choice(['☀️', '🌧️', '❄️'])}")

print(f"median   = {statistics.median([1, 2, 3, 4, 100])}")  # robust to outliers

sqrt(2)  = 1.4142
pi       = 3.141593
random   = 🌧️
median   = 3


In [14]:
# The data-science "Big 3" — we will use them in Notebooks 7–9
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"numpy      version: {np.__version__}")
print(f"pandas     version: {pd.__version__}")
print(f"matplotlib version: {plt.matplotlib.__version__}")

numpy      version: 2.0.2
pandas     version: 2.3.3
matplotlib version: 3.9.4


## 11. Putting it together — a tiny data-cleaning toolkit

Three small functions composed into a pipeline. Each does **one** thing well.

In [15]:
from typing import Iterable, Optional

def parse_number(text: str) -> Optional[float]:
    """Try to convert a string to float; return None when it cannot."""
    try:
        return float(text)
    except (ValueError, TypeError):
        return None

def clean(values: Iterable, min_valid: float = -1e9, max_valid: float = 1e9) -> list:
    """Drop non-numeric values and those outside [min_valid, max_valid]."""
    out = []
    for v in values:
        x = parse_number(v) if isinstance(v, str) else v
        if x is None:
            continue
        if not (min_valid <= x <= max_valid):
            continue
        out.append(float(x))
    return out

def summary(values: Iterable[float]) -> dict:
    """Compute n, min, max, mean, std for a list of numbers."""
    values = list(values)
    n = len(values)
    if n == 0:
        return {"n": 0}
    mean = sum(values) / n
    var  = sum((x - mean)**2 for x in values) / n
    return {"n": n, "min": min(values), "max": max(values), "mean": mean, "std": var ** 0.5}

# Compose them on messy data
raw = ["18.5", "21.0", "n/a", "19.7", None, "22.3", "", "500000", "-273.15", "20.4"]
clean_data = clean(raw, min_valid=-50, max_valid=60)

print(f"raw   ({len(raw):>2}): {raw}")
print(f"clean ({len(clean_data):>2}): {clean_data}\n")

for k, v in summary(clean_data).items():
    if isinstance(v, float):
        print(f"  {k:<5}: {v:.3f}")
    else:
        print(f"  {k:<5}: {v}")

raw   (10): ['18.5', '21.0', 'n/a', '19.7', None, '22.3', '', '500000', '-273.15', '20.4']
clean ( 5): [18.5, 21.0, 19.7, 22.3, 20.4]

  n    : 5
  min  : 18.500
  max  : 22.300
  mean : 20.380
  std  : 1.270


**Notice what we did.** Each function has one job (parse / clean / summarise) and a docstring. We can swap any of them later without touching the others. This is the *modular* mindset that makes large codebases manageable.

## 🧪 Practice exercises

### Exercise 5.1 — Temperature converters

Write `c_to_f(c)` and `f_to_c(f)` with docstrings. Verify them on a few values (`0°C → 32°F`, `100°C → 212°F`, `-40°C → -40°F`).

In [16]:
# Your code here  👇


<details>
<summary>💡 Click to reveal the solution</summary>

```python
def c_to_f(c: float) -> float:
    "Convert Celsius to Fahrenheit."
    return c * 9 / 5 + 32

def f_to_c(f: float) -> float:
    "Convert Fahrenheit to Celsius."
    return (f - 32) * 5 / 9

for c in [-40, 0, 25, 100]:
    f = c_to_f(c)
    print(f"{c:>4}°C ↔ {f:>5.1f}°F  (round-trip: {f_to_c(f):.1f}°C)")
```
</details>

### Exercise 5.2 — Stats function

Write `stats(values)` returning a dict with `n`, `mean`, `median`, `std`. Implement it yourself with only `sum`, `len`, `sorted`, and arithmetic (no `statistics` module).

In [17]:
# Your code here  👇


<details>
<summary>💡 Click to reveal the solution</summary>

```python
def stats(values):
    n = len(values)
    if n == 0:
        return {"n": 0, "mean": None, "median": None, "std": None}

    mean = sum(values) / n
    s = sorted(values)
    if n % 2 == 1:
        median = s[n // 2]
    else:
        median = (s[n // 2 - 1] + s[n // 2]) / 2

    var = sum((x - mean) ** 2 for x in values) / n
    return {"n": n, "mean": mean, "median": median, "std": var ** 0.5}

print(stats([2, 4, 4, 4, 5, 5, 7, 9]))
```
</details>

### Exercise 5.3 — Higher-order function

Write `keep_if(values, predicate)` that returns only items where `predicate(item)` is true. Test it with two different lambdas.

In [18]:
# Your code here  👇


<details>
<summary>💡 Click to reveal the solution</summary>

```python
def keep_if(values, predicate):
    return [v for v in values if predicate(v)]

nums = [1, -2, 3, -4, 5, -6, 7]
print(keep_if(nums, lambda x: x > 0))          # positives
print(keep_if(nums, lambda x: x % 2 == 0))     # evens
print(keep_if(nums, lambda x: abs(x) >= 3))    # large in magnitude
```

Passing a function as an argument is called a **higher-order function**. Built-ins `filter`, `map`, and `sorted(key=...)` are all examples.
</details>

### Exercise 5.4 — Default-argument pitfall

What does this print after the third call? Predict, then run.

In [19]:
def add(item, bag=[]):
    bag.append(item)
    return bag

print(add("a"))
print(add("b"))
print(add("c"))

['a']
['a', 'b']
['a', 'b', 'c']


<details>
<summary>💡 Click to reveal the solution</summary>

Output is `['a']`, `['a', 'b']`, `['a', 'b', 'c']` — the default list is **shared across calls**.

**Why?** The default value is evaluated **once**, when Python reads the `def` line — not freshly on every call. The one list object is stored on the function and reused by every call that omits `bag`. You can even see it sitting there: after the three calls, `add.__defaults__` is `(['a', 'b', 'c'],)`.

Fix:

```python
def add(item, bag=None):
    if bag is None:
        bag = []
    bag.append(item)
    return bag
```

The `None` sentinel + create-inside is the standard Python idiom: `None` is immutable, and the fresh `[]` is created *per call*, inside the body.
</details>

### Exercise 5.5 — Debug me 🐞

Find the bug — `average` should return the average but is consistently off.

In [20]:
def average(values):
    total = 0
    for v in values:
        total = v          # bug!
    return total / len(values)

print(average([10, 20, 30, 40, 50]))   # expected: 30.0

10.0


<details>
<summary>💡 Click to reveal the solution</summary>

`total = v` overwrites instead of accumulates. Should be `total += v`:

```python
def average(values):
    if not values:
        return None
    total = 0
    for v in values:
        total += v
    return total / len(values)
```
</details>

## 🎁 Bonus mini-project — Modular grade-report generator

Build three small functions:

1. `grade(score)` returns `"A"/"B"/"C"/"D"/"F"`.
2. `student_summary(student)` takes `{"name": ..., "scores": [...]}` and returns a dict with `name`, `mean`, `grade`.
3. `class_report(students)` prints a tidy table plus the class average.

Use docstrings throughout.

In [21]:
# Your code here  👇
students = [
    {"name": "Alice",   "scores": [85, 92, 78]},
    {"name": "Bob",     "scores": [70, 65, 72]},
    {"name": "Charlie", "scores": [95, 88, 91]},
    {"name": "Diana",   "scores": [55, 60, 58]},
]

<details>
<summary>💡 Click to reveal the solution</summary>

```python
def grade(score):
    "Map a numeric score (0-100) to a letter grade."
    if score >= 90: return "A"
    if score >= 80: return "B"
    if score >= 70: return "C"
    if score >= 60: return "D"
    return "F"

def student_summary(student):
    "Return {name, mean, grade} for one student."
    mean = sum(student["scores"]) / len(student["scores"])
    return {"name": student["name"], "mean": mean, "grade": grade(mean)}

def class_report(students):
    "Print a table of per-student summaries plus the class average."
    summaries = [student_summary(s) for s in students]
    print(f"{'Name':<10}{'Mean':>8}{'Grade':>8}")
    print("-" * 26)
    for s in summaries:
        print(f"{s['name']:<10}{s['mean']:>8.2f}{s['grade']:>8}")
    class_mean = sum(s['mean'] for s in summaries) / len(summaries)
    print("-" * 26)
    print(f"{'CLASS':<10}{class_mean:>8.2f}{grade(class_mean):>8}")

class_report(students)
```
</details>

## 🧠 Key takeaways

1. A function bundles reusable logic behind a meaningful name — one place to fix, instead of diverging copies.
2. `return` hands a value back for further computation; `print` only displays it. A function without `return` gives you `None`.
3. Parameters can have **defaults** and be passed by **position** or **keyword**.
4. `*args` and `**kwargs` collect extra positional and keyword arguments — that is what those symbols mean in the `plt.plot` and `pd.read_csv` signatures. In a *call*, `*`/`**` unpack sequences/dicts instead.
5. Return one value, a tuple, or a dictionary if several values have distinct meanings.
6. Variables inside a function are **local** (they die when the call ends — `return` is the door out); passing mutable objects gives the function the power to mutate them.
7. Document with **docstrings** and (optionally) **type hints**.
8. **Lambdas** are short single-expression functions you pass to other functions.
9. `import` brings in code from the standard library and third-party packages.

## ✅ Self-assessment

- [ ] Define a function with a docstring and at least one default argument.
- [ ] Explain why a variable assigned from a function that only `print`s ends up holding `None`.
- [ ] Return more than one value and unpack the result.
- [ ] Use `*args` and `**kwargs` correctly.
- [ ] Spot the mutable-default trap.
- [ ] Pass a lambda to `sorted` to sort by a chosen field.

## 🚀 Next step

Continue with **[Notebook 6 — NumPy Fundamentals](06_numpy_fundamentals.ipynb)**, where you will meet the array — the data structure underneath every machine-learning library, and underneath the pandas tables you will build in Notebook 7.

*(Previous notebook: [04_dictionaries_advanced.ipynb](04_dictionaries_advanced.ipynb))*
